# Deteksi Hoaks di Media Sosial — Proyek Akhir Machine Learning
**Kategori:** NLP / Supervised Learning (Klasifikasi Teks Biner: Hoaks vs Valid)

Notebook ini mencakup seluruh tahapan wajib:
1. Studi Literatur (ringkasan di Laporan Akhir)
2. Pengumpulan Data
3. Data Preprocessing
4. Pembangunan Model (perbandingan >=2 algoritma untuk bonus nilai)
5. Evaluasi Model
6. Menyimpan model untuk deployment ke Streamlit

Jalankan sel demi sel di Google Colab (Runtime > Run all setelah upload dataset).

## 1. Instalasi & Import Library

In [ ]:
!pip install Sastrawi -q

import pandas as pd
import numpy as np
import re
import string
import pickle
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

np.random.seed(42)

## 2. Pengumpulan Data

Dataset publik: **Indonesia False News (Hoax) Dataset** oleh Muhammad Ghazi Muharam (Kaggle),
berisi kumpulan status/berita berbahasa Indonesia berlabel `label` (hoax / valid) — total ribuan baris data,
sudah memenuhi ketentuan minimal 500 dokumen/teks untuk kategori NLP.

Sumber: https://www.kaggle.com/datasets/muhammadghazimuharam/indonesiafalsenews

**Cara mengambil dataset di Colab (pilih salah satu):**

**Opsi A — Kaggle API (disarankan):**
```python
from google.colab import files
files.upload()  # upload kaggle.json (API token dari akun Kaggle Anda)
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d muhammadghazimuharam/indonesiafalsenews
!unzip -q indonesiafalsenews.zip -d data/
```

**Opsi B — Upload manual CSV** yang sudah Anda unduh dari Kaggle ke Colab (drag & drop ke panel Files),
lalu sesuaikan `DATA_PATH` di bawah.

In [ ]:
DATA_PATH = "data/Data_latih.csv"  # sesuaikan dengan nama file hasil unduhan/ekstraksi Anda

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

> Catatan: nama kolom teks dan label bisa berbeda tergantung versi dataset yang Anda unduh
> (mis. `text`/`narasi` untuk isi berita, `label` untuk kelas hoax/valid).
> Sesuaikan `TEXT_COL` dan `LABEL_COL` di sel berikut agar cocok dengan file Anda.

In [ ]:
TEXT_COL = "narasi"
LABEL_COL = "label"

df = df[[TEXT_COL, LABEL_COL]].dropna().drop_duplicates().reset_index(drop=True)
df[LABEL_COL] = df[LABEL_COL].astype(str).str.lower().map(
    lambda x: 1 if x in ["1", "hoax", "hoaks", "fake"] else 0
)
print(df[LABEL_COL].value_counts())
print("Total data:", len(df))

## 3. Data Preprocessing

Tahapan: case folding -> cleaning (URL, mention, angka, tanda baca) -> tokenizing -> stopword removal -> stemming (Sastrawi).

In [ ]:
stemmer = StemmerFactory().create_stemmer()
stopword_remover = StopWordRemoverFactory().create_stop_word_remover()

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"@\w+|#\w+", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

def preprocess(text):
    text = clean_text(text)
    text = stopword_remover.remove(text)
    text = stemmer.stem(text)
    return text

# Untuk dataset besar, stemming Sastrawi cukup lambat — proses bertahap dengan progress
from tqdm.auto import tqdm
tqdm.pandas()
df["clean_text"] = df[TEXT_COL].astype(str).progress_apply(preprocess)
df[["clean_text", LABEL_COL]].head()

## 4. Split Data & TF-IDF Vectorization

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"], df[LABEL_COL], test_size=0.2, random_state=42, stratify=df[LABEL_COL]
)

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Ukuran train:", X_train_tfidf.shape, " | Ukuran test:", X_test_tfidf.shape)

## 5. Pembangunan & Perbandingan Model

Melatih 3 algoritma (memenuhi bonus nilai: perbandingan >= 2 algoritma):
- Multinomial Naive Bayes
- Logistic Regression
- Linear SVM

In [ ]:
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Linear SVM": LinearSVC(),
}

results = {}
for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    results[name] = {
        "model": model,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "y_pred": y_pred,
    }
    print(f"--- {name} ---")
    print(classification_report(y_test, y_pred, target_names=["Valid", "Hoax"]))

## 6. Evaluasi Model — Perbandingan & Confusion Matrix

In [ ]:
summary = pd.DataFrame({
    name: {k: v for k, v in r.items() if k not in ["model", "y_pred"]}
    for name, r in results.items()
}).T
summary = summary.sort_values("f1", ascending=False)
print(summary)

fig, axes = plt.subplots(1, len(models), figsize=(5 * len(models), 4))
for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, r["y_pred"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Valid", "Hoax"], yticklabels=["Valid", "Hoax"])
    ax.set_title(name)
    ax.set_xlabel("Prediksi")
    ax.set_ylabel("Aktual")
plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150)
plt.show()

## 7. Menyimpan Model Terbaik untuk Deployment Streamlit

In [ ]:
best_name = summary.index[0]
best_model = results[best_name]["model"]
print("Model terbaik:", best_name)

with open("model_hoax.pkl", "wb") as f:
    pickle.dump(best_model, f)

with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

print("Tersimpan: model_hoax.pkl, tfidf_vectorizer.pkl")
print("Unduh kedua file ini lalu letakkan di folder project Streamlit Anda (sejajar dengan app.py).")

## 8. (Opsional) Uji Coba Prediksi Manual

In [ ]:
def predict_hoax(text, model=best_model, vectorizer=tfidf):
    clean = preprocess(text)
    vec = vectorizer.transform([clean])
    pred = model.predict(vec)[0]
    return "HOAX" if pred == 1 else "VALID"

contoh = "Bagikan pesan ini ke 10 orang atau HP Anda akan diretas hacker"
print(predict_hoax(contoh))